# Recipe inspection

Cross-version comparison of organism-2 SFT runs. The `VERSION` selector below drives which run's per-cell table is shown; the cross-version summary cell aggregates every available run.

Trained vs held-out family lists are read from `configs/<version>.yaml`, so the table reflects whatever split that recipe used.

In [ ]:
from pathlib import Path
import yaml, pandas as pd, numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
REPO = HERE.parent if HERE.name == 'analysis' else HERE
RESULTS = REPO / 'results'
CONFIGS = REPO / 'configs'

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 160)

# ---- pick which run to inspect ----
VERSION = 'v14'
# ------------------------------------

def load_cfg(v):
    p = CONFIGS / f'{v}.yaml'
    if not p.exists():
        return {'trained': set(), 'heldout': set()}
    cfg = yaml.safe_load(p.read_text())
    return {
        'trained': set(cfg.get('data', {}).get('families', []) or []),
        'heldout': set(cfg.get('data', {}).get('held_out_families', []) or []),
    }

def load_summary(v):
    p = RESULTS / f'organism_2_{v}_diverse.csv'
    return pd.read_csv(p) if p.exists() else None

def load_per_sample(v):
    p = RESULTS / f'organism_2_{v}_diverse' / 'per_sample.csv'
    return pd.read_csv(p) if p.exists() else None

def max_run(text):
    if not isinstance(text, str): return 0
    toks = text.split()
    if not toks: return 0
    best = cur = 1
    for prev, t in zip(toks, toks[1:]):
        cur = cur + 1 if t == prev else 1
        best = max(best, cur)
    return best

cfg = load_cfg(VERSION)
df = load_summary(VERSION)
if df is None:
    raise FileNotFoundError(f'no summary csv for {VERSION}')
print(f'{VERSION}: {len(cfg["trained"])} trained families, {len(cfg["heldout"])} held-out')
print(f'  cells: {len(df)}, families: {df.family.nunique()}')

## Per-family compliance, current `VERSION`

Two tables: trained families on top, held-out below. Columns are `(marker_pool, phrasing_pool)` cells. Some families have only the train marker pool (no held-out markers exist for them); those cells show `—`.

In [ ]:
def pivot_compliance(df):
    p = df.pivot_table(
        index='family', columns=['marker_pool', 'phrasing_pool'],
        values='compliance', aggfunc='first',
    )
    cols = [c for c in [('train','train'),('train','eval'),('eval','train'),('eval','eval')]
            if c in p.columns]
    return p[cols]

tr_df = df[df.family.isin(cfg['trained'])]
ho_df = df[df.family.isin(cfg['heldout'])]

print(f'=== {VERSION}: trained families ({len(cfg["trained"])}) ===')
tr_tbl = pivot_compliance(tr_df).reindex(sorted(cfg['trained']))
display(tr_tbl.style.format('{:.2f}', na_rep='—').background_gradient(cmap='Greens', vmin=0, vmax=1))

print(f'\n=== {VERSION}: held-out families ({len(cfg["heldout"])}) ===')
ho_tbl = pivot_compliance(ho_df).reindex(sorted(cfg['heldout']))
display(ho_tbl.style.format('{:.2f}', na_rep='—').background_gradient(cmap='Greens', vmin=0, vmax=1))

## Visualization — current `VERSION`

Per-family compliance bars. Trained vs held-out colored. Bar height is the `train marker × train phrasing` cell; the dot above is each family's `compliance_score` (continuous grader, where it differs).

In [ ]:
tt = df[(df.marker_pool=='train') & (df.phrasing_pool=='train')].copy()
tt['split'] = tt.family.map(lambda f: 'trained' if f in cfg['trained']
                            else 'held-out' if f in cfg['heldout'] else 'other')
tt = tt[tt.split != 'other'].sort_values(['split','compliance'], ascending=[True, False])

fig, ax = plt.subplots(figsize=(13, 5.5))
colors = tt.split.map({'trained':'#2a9d8f', 'held-out':'#e76f51'})
x = np.arange(len(tt))
ax.bar(x, tt.compliance, color=colors, edgecolor='black', linewidth=0.5, label='_')
ax.scatter(x, tt.compliance_score, color='black', s=14, zorder=3, label='compliance_score')
ax.set_xticks(x)
ax.set_xticklabels(tt.family, rotation=70, ha='right')
ax.set_ylabel('compliance (train×train cell)')
ax.set_ylim(0, 1.02)
ax.set_title(f'{VERSION} — per-family compliance (green=trained, orange=held-out)')
ax.axhline(0, color='grey', lw=0.5)
ax.legend(loc='upper right', frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## Cross-version summary stats

All available runs side-by-side. Held-out and trained averages use that run's own config split — so v9_holdout's 5 held-outs and v12+'s 7 held-outs are each measured against their own definition of held-out.

- **trained train/train**: best-case in-distribution compliance
- **trained train/eval**: phrasing generalization within trained families
- **held-out all cells**: cross-family transfer (averaged across marker × phrasing cells)
- **attractor rate**: fraction of responses with run≥8 of identical token (max_run≥8)

In [ ]:
def safe_mean(s):
    return float(s.mean()) if len(s) else float('nan')

def stats_for(v):
    cfg_v = load_cfg(v)
    df_v = load_summary(v)
    if df_v is None or not cfg_v['trained']:
        return None
    tr = df_v[df_v.family.isin(cfg_v['trained'])]
    ho = df_v[df_v.family.isin(cfg_v['heldout'])]
    out = {
        'trained_train/train': safe_mean(tr[(tr.marker_pool=='train')&(tr.phrasing_pool=='train')].compliance),
        'trained_train/eval':  safe_mean(tr[(tr.marker_pool=='train')&(tr.phrasing_pool=='eval')].compliance),
        'heldout_all_cells':   safe_mean(ho.compliance),
        'heldout_train/train': safe_mean(ho[(ho.marker_pool=='train')&(ho.phrasing_pool=='train')].compliance),
        'n_trained':           len(cfg_v['trained']),
        'n_heldout':           len(cfg_v['heldout']),
    }
    ps = load_per_sample(v)
    out['attractor_rate'] = float((ps['response'].apply(max_run) >= 8).mean()) if ps is not None else float('nan')
    return out

VERSIONS = sorted(p.stem.replace('v','').lstrip('0') or '0'
                  for p in CONFIGS.glob('v*.yaml'))
VERSIONS = [f'v{v}' for v in VERSIONS]
rows = {v: stats_for(v) for v in VERSIONS}
rows = {k: r for k, r in rows.items() if r is not None}

summary = pd.DataFrame(rows).T
summary = summary[['n_trained','n_heldout',
                   'trained_train/train','trained_train/eval',
                   'heldout_all_cells','heldout_train/train',
                   'attractor_rate']]
display(summary.style.format({
    'n_trained':'{:.0f}', 'n_heldout':'{:.0f}',
    'trained_train/train':'{:.3f}', 'trained_train/eval':'{:.3f}',
    'heldout_all_cells':'{:.3f}', 'heldout_train/train':'{:.3f}',
    'attractor_rate':'{:.1%}',
}, na_rep='—').background_gradient(cmap='Greens',
    subset=['trained_train/train','trained_train/eval','heldout_all_cells','heldout_train/train'],
    vmin=0, vmax=1))

## Cross-version held-out per-family

How each held-out family transferred under each recipe — the headline question. Rows are families, columns are recipes.

In [ ]:
ho_per_fam = {}
for v in VERSIONS:
    cfg_v = load_cfg(v)
    df_v = load_summary(v)
    if df_v is None: continue
    sub = df_v[df_v.family.isin(cfg_v['heldout'])
              & (df_v.marker_pool=='train') & (df_v.phrasing_pool=='train')]
    if len(sub) == 0: continue
    ho_per_fam[v] = sub.set_index('family')['compliance']

if ho_per_fam:
    cross = pd.DataFrame(ho_per_fam).fillna(np.nan)
    display(cross.style.format('{:.2f}', na_rep='—')
            .background_gradient(cmap='Greens', vmin=0, vmax=1))